[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/00_python_essentials/00_python_essentials_solutions.ipynb)

# 00. NumPy / Pandas / PyTorch 필수 라이브러리 실습 — 연습 문제 해설

[00_python_essentials.ipynb](00_python_essentials.ipynb)의 연습 문제 정답입니다. 먼저 직접 풀어본 뒤 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q numpy pandas torch
import numpy as np
import pandas as pd
import torch


## NumPy 연습 문제 해설

**1. `(5, 5)` 항등 행렬을 만들고 모든 원소에 3을 곱한 뒤 행 단위 합 구하기**

In [ ]:
m = np.eye(5) * 3
row_sums = m.sum(axis=1)
print(m)
print("행 단위 합:", row_sums)   # 대각 원소만 3이므로 모든 행의 합은 3


**2. 0~99 정수 배열에서 50 이상인 값만 골라 평균 구하기**

In [ ]:
rng = np.random.default_rng(42)
arr = rng.integers(0, 100, size=20)
filtered = arr[arr >= 50]
print("전체:", arr)
print("50 이상:", filtered)
print("평균:", filtered.mean() if filtered.size else "해당 값 없음")


**3. `(4, 3)` × `(3, 2)` 행렬곱, 순서를 바꾸면 오류가 나는 이유**

In [ ]:
A = np.arange(12).reshape(4, 3)
B = np.arange(6).reshape(3, 2)

print("A @ B shape:", (A @ B).shape)   # (4,3) @ (3,2) -> (4,2), 내부 차원 3이 일치

try:
    B @ A
except ValueError as e:
    print("B @ A 오류:", e)
    # B의 열 개수(2)와 A의 행 개수(4)가 달라서 행렬곱 정의(내부 차원 일치)를 만족하지 못함


## Pandas 연습 문제 해설

In [ ]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "score": [88, 92, 79, 65, 95],
    "team": ["A", "B", "A", "B", "A"],
})
df["passed"] = df["score"] >= 70
df


**1. `age` 열 추가 후 나이순 정렬**

In [ ]:
df["age"] = [23, 31, 27, 22, 35]
df_by_age = df.sort_values("age")
df_by_age


**2. 팀별로 `passed`가 `True`인 인원 수**

In [ ]:
passed_count = df.groupby("team")["passed"].sum()
print(passed_count)


**3. 팀 평균보다 점수가 높은 사람만 골라내기**

In [ ]:
team_avg = df.groupby("team")["score"].transform("mean")
above_avg = df[df["score"] > team_avg]
above_avg


## PyTorch 연습 문제 해설

**1. `y = x**2 + 2x`의 미분을 `.backward()`로 구하고 손 계산과 비교**

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x
y.backward()

print("autograd dy/dx:", x.grad.item())
print("손 계산 dy/dx = 2x + 2:", 2 * 3.0 + 2)   # 두 값이 일치해야 함


**2. `W.grad.zero_()`를 지우면 벌어지는 일**

gradient는 기본적으로 **누적(accumulate)**됩니다. `zero_()`로 초기화하지 않으면 이전 epoch의
gradient가 계속 더해져서, `W -= lr * W.grad`가 실제보다 훨씬 큰 값을 빼게 되어 학습이 발산합니다.

In [ ]:
x_data = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_data = torch.tensor([5.0, 7.0, 9.0, 11.0])

W = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.01

for epoch in range(10):
    y_pred = W * x_data + b
    cost = ((y_pred - y_data) ** 2).mean()
    cost.backward()
    with torch.no_grad():
        W -= lr * W.grad
        b -= lr * b.grad
        # zero_()를 호출하지 않음 -> gradient가 계속 누적됨
    print(f"epoch {epoch}  cost={cost.item():.4f}  W={W.item():.3f}  W.grad={W.grad.item():.3f}")
    # cost가 줄어들지 않고 W.grad 값이 매 epoch 비정상적으로 커지는 것을 확인할 수 있음


**3. `Tensor` <-> `ndarray` 왕복 변환**

In [ ]:
np_arr = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
t = torch.from_numpy(np_arr)     # ndarray -> Tensor (메모리 공유)
back_to_np = t.numpy()           # Tensor -> ndarray (메모리 공유)

print(type(t), t)
print(type(back_to_np), back_to_np)

t[0, 0] = 99.0                   # t를 바꾸면 np_arr도 같이 바뀜 (메모리를 공유하기 때문)
print("np_arr도 변경됨:\n", np_arr)
